# Emotion Detection: Model Training & Evaluation

This notebook trains and evaluates **7 models** on **2 datasets** for emotion detection:

**Models:**
1. DistilBERT
2. DeBERTa
3. RoBERTa
4. BanglaBERT
5. BanglishBERT
6. XLNet
7. Falcon-7B (Quantized with QLoRA)

**Datasets:**
1. Empathetic Dialogue (32 emotion classes)
2. Daily Dialogue (7 emotion classes)

**Metrics:** F1, Precision, Recall, Accuracy, Confusion Matrix

## Setup & Installation

In [20]:
# Install required packages (GPU-enabled Torch + NLP stack)
%pip install --upgrade torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124
%pip install transformers datasets accelerate peft bitsandbytes -q
%pip install scikit-learn pandas matplotlib seaborn -q
# For XLNet tokenizer
%pip install sentencepiece -q  

Looking in indexes: https://download.pytorch.org/whl/cu124
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [21]:
import os
import json
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    classification_report, confusion_matrix
)
from sklearn.utils.class_weight import compute_class_weight

from datasets import Dataset
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, DataCollatorWithPadding,
    BitsAndBytesConfig
)
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training

import warnings
warnings.filterwarnings('ignore')

# Set device-related env vars early
os.environ.setdefault('CUDA_VISIBLE_DEVICES', '0')
os.environ['PYTORCH_MPS_HIGH_WATERMARK_RATIO'] = '0.0'

# Check Device (GPU/MPS/CPU) and log useful diagnostics
if torch.cuda.is_available():
    torch.cuda.init()
    device = torch.device('cuda')
    gpu_idx = torch.cuda.current_device()
    props = torch.cuda.get_device_properties(gpu_idx)
    capability = props.major + props.minor / 10
    total_mem_gb = props.total_memory / (1024 ** 3)
    print(f"Using GPU: {props.name} (capability {capability:.1f}, {total_mem_gb:.1f} GB)")
    print(f"PyTorch CUDA build: {torch.version.cuda} | Torch version: {torch.__version__}")
    print(f"Visible CUDA devices: {torch.cuda.device_count()}")
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
elif torch.backends.mps.is_available():
    device = torch.device('mps')
    print("Using MPS (Apple Silicon GPU)")
else:
    device = torch.device('cpu')
    print("Using CPU - install CUDA drivers + GPU build of PyTorch for acceleration")

Using GPU: NVIDIA GeForce RTX 3050 Laptop GPU (capability 8.6, 4.0 GB)
PyTorch CUDA build: 12.4 | Torch version: 2.6.0+cu124
Visible CUDA devices: 1


## Configuration

In [22]:
# ========== CONFIGURATION ==========
# Set which dataset to use: 'empathetic' or 'daily'
DATASET_NAME = 'empathetic'  # Change to 'daily' for Daily Dialogue

# Model configurations
MODELS = {
    'distilbert': 'distilbert-base-uncased',
    'deberta': 'microsoft/deberta-base',
    'roberta': 'roberta-base',
    'banglabert': 'csebuetnlp/banglabert',
    'banglishbert': 'csebuetnlp/banglishbert',  # Available Banglish model
    'xlnet': 'xlnet-base-cased',
    'falcon': 'tiiuae/falcon-7b'
}

# Training hyperparameters
MAX_LENGTH = 128
BATCH_SIZE = 16  # Reduce for larger models
LEARNING_RATE = 2e-5
NUM_EPOCHS = 3
WEIGHT_DECAY = 0.01

# Precision controls
# DEFAULT_PRECISION: 'auto' tries bf16 -> fp16 -> fp32 based on hardware.
DEFAULT_PRECISION = 'auto'
MODEL_PRECISION_OVERRIDES = {
    # DeBERTa fails in pure fp16 due to masked_fill underflow (see CUDA error).
    'deberta': 'fp32',
}

# Output directory
OUTPUT_DIR = './results'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Dataset: {DATASET_NAME}")
print(f"Output directory: {OUTPUT_DIR}")

Dataset: empathetic
Output directory: ./results


## Load Dataset

In [23]:
def load_dataset_splits(dataset_name):
    """Load preprocessed train/val/test splits."""
    if dataset_name == 'empathetic':
        train_df = pd.read_csv('Datasets/empathetic_train.csv')
        val_df = pd.read_csv('Datasets/empathetic_val.csv')
        test_df = pd.read_csv('Datasets/empathetic_test.csv')
        label_map = pd.read_csv('Datasets/empathetic_label_map.csv')
        num_labels = len(label_map)
    else:  # daily
        train_df = pd.read_csv('Datasets/daily_train.csv')
        val_df = pd.read_csv('Datasets/daily_val.csv')
        test_df = pd.read_csv('Datasets/daily_test.csv')
        label_map = pd.read_csv('Datasets/daily_label_map.csv')
        num_labels = 7
    
    return train_df, val_df, test_df, num_labels, label_map

# Load data
train_df, val_df, test_df, num_labels, label_map = load_dataset_splits(DATASET_NAME)
print(f"Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")
print(f"Number of labels: {num_labels}")
print(f"\nLabel distribution in training set:")
print(train_df['label_encoded'].value_counts().sort_index())

Train: 51672, Val: 6459, Test: 6460
Number of labels: 43

Label distribution in training set:
label_encoded
6     1675
7     1837
8     1770
9     1621
10    1630
11    1239
12    1355
13    1412
14    1630
15    1522
16    1485
17    1575
18    1635
19    1475
20    1972
21    1027
22    1636
23    1673
24    1642
25    1615
26    1603
27    1564
28    1562
29    1685
31    1597
32    1550
33    1798
34    1770
35    1418
36    2636
41    1659
42    1404
Name: count, dtype: int64


In [24]:
# Compute class weights for imbalanced data
val_classes = np.unique(train_df['label_encoded'])
computed_weights = compute_class_weight(
    class_weight='balanced',
    classes=val_classes,
    y=train_df['label_encoded']
)

# Create a weights tensor for all labels (handling missing/filtered classes)
class_weights = np.ones(num_labels, dtype=np.float32)
for cls_idx, weight in zip(val_classes, computed_weights):
    class_weights[cls_idx] = weight

class_weights = torch.tensor(class_weights, dtype=torch.float32).to(device)
print(f"Class weights computed for {len(class_weights)} classes (Total labels: {num_labels})")

Class weights computed for 43 classes (Total labels: 43)


## Helper Functions

In [25]:
def tokenize_data(df, tokenizer, max_length=128):
    """Tokenize text data for training."""
    dataset = Dataset.from_pandas(df[['text', 'label_encoded']].rename(columns={'label_encoded': 'labels'}))
    
    def tokenize_function(examples):
        return tokenizer(
            examples['text'],
            padding='max_length',
            truncation=True,
            max_length=max_length
        )
    
    tokenized = dataset.map(tokenize_function, batched=True)
    tokenized = tokenized.remove_columns(['text'])
    tokenized.set_format('torch')
    return tokenized


def compute_metrics(eval_pred):
    """Compute evaluation metrics."""
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    
    precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average='weighted')
    accuracy = accuracy_score(labels, predictions)
    
    return {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1
    }


def plot_confusion_matrix(y_true, y_pred, labels, title, save_path=None):
    """Plot and save confusion matrix."""
    cm = confusion_matrix(y_true, y_pred)
    
    plt.figure(figsize=(12, 10))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title(title)
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    plt.close()


def save_results(model_name, dataset_name, metrics, report, save_dir):
    """Save evaluation results to JSON."""
    results = {
        'model': model_name,
        'dataset': dataset_name,
        'timestamp': datetime.now().isoformat(),
        'metrics': metrics,
        'classification_report': report
    }
    
    filename = f"{save_dir}/{model_name}_{dataset_name}_results.json"
    with open(filename, 'w') as f:
        json.dump(results, f, indent=2)
    print(f"Results saved to {filename}")
    return results

## Custom Trainer with Class Weights

In [26]:
class WeightedTrainer(Trainer):
    """Custom Trainer that uses class weights for loss computation."""
    
    def __init__(self, class_weights, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights
    
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop('labels')
        outputs = model(**inputs)
        logits = outputs.logits
        
        # Ensure class_weights is on the same device as logits
        if self.class_weights.device != logits.device:
            self.class_weights = self.class_weights.to(logits.device)
        
        loss_fct = torch.nn.CrossEntropyLoss(weight=self.class_weights)
        loss = loss_fct(logits, labels)
        
        return (loss, outputs) if return_outputs else loss

## Training Function

In [27]:
def train_and_evaluate_model(model_name, model_path, train_df, val_df, test_df, 
                              num_labels, class_weights, dataset_name, 
                              use_quantization=False):
    """
    Train and evaluate a single model.
    
    Args:
        model_name: Short name for the model
        model_path: HuggingFace model path
        train_df, val_df, test_df: Data splits
        num_labels: Number of classification labels
        class_weights: Tensor of class weights
        dataset_name: 'empathetic' or 'daily'
        use_quantization: Whether to use 4-bit quantization (for Falcon)
    
    Returns:
        dict: Evaluation results
    """
    print(f"\n{'='*60}")
    print(f"Training: {model_name.upper()} on {dataset_name.upper()}")
    print(f"{'='*60}")
    
    # Clear GPU memory
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    elif torch.backends.mps.is_available():
        torch.mps.empty_cache()
    
    # Load tokenizer
    tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
    
    # Handle missing pad token
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    
    # Tokenize datasets
    print("Tokenizing data...")
    train_dataset = tokenize_data(train_df, tokenizer, MAX_LENGTH)
    val_dataset = tokenize_data(val_df, tokenizer, MAX_LENGTH)
    test_dataset = tokenize_data(test_df, tokenizer, MAX_LENGTH)
    
    # Load model
    print("Loading model...")
    if use_quantization:
        # 4-bit quantization for Falcon-7B
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True
        )
        
        model = AutoModelForSequenceClassification.from_pretrained(
            model_path,
            num_labels=num_labels,
            quantization_config=bnb_config,
            device_map="auto",
            trust_remote_code=True
        )
        
        # Prepare for LoRA
        model = prepare_model_for_kbit_training(model)
        
        # Configure LoRA
        lora_config = LoraConfig(
            r=8,
            lora_alpha=32,
            target_modules=["query_key_value"],
            lora_dropout=0.05,
            bias="none",
            task_type=TaskType.SEQ_CLS
        )
        model = get_peft_model(model, lora_config)
        model.print_trainable_parameters()
        
        # Adjust batch size for large model
        batch_size = 4
    else:
        model = AutoModelForSequenceClassification.from_pretrained(
            model_path,
            num_labels=num_labels,
            trust_remote_code=True
        )
        model.to(device)
        batch_size = BATCH_SIZE
    
    # Precision selection: prefer bf16 -> fp16 -> fp32 unless overridden
    def resolve_precision(model_key, quantized):
        if quantized:
            return False, False, "fp32 (precision handled inside 4-bit adapter)"
        pref = MODEL_PRECISION_OVERRIDES.get(model_key, DEFAULT_PRECISION)
        bf16_supported = torch.cuda.is_available() and getattr(torch.cuda, "is_bf16_supported", lambda: False)()
        fp16_supported = torch.cuda.is_available()
        use_fp16 = False
        use_bf16 = False
        label = "fp32"
        if pref == 'fp32':
            label = "fp32"
        elif pref == 'fp16':
            if fp16_supported:
                use_fp16 = True
                label = "fp16"
            else:
                label = "fp32 (fp16 unsupported, fallback)"
        elif pref == 'bf16':
            if bf16_supported:
                use_bf16 = True
                label = "bf16"
            else:
                label = "fp32 (bf16 unsupported, fallback)"
        else:  # auto preference
            if bf16_supported:
                use_bf16 = True
                label = "bf16 (auto)"
            elif fp16_supported:
                use_fp16 = True
                label = "fp16 (auto)"
            else:
                label = "fp32 (auto)"
        return use_fp16, use_bf16, label
    
    use_fp16, use_bf16, precision_label = resolve_precision(model_name, use_quantization)
    print(f"Precision mode: {precision_label}")
    
    # Handle pad token for model
    if model.config.pad_token_id is None:
        model.config.pad_token_id = tokenizer.pad_token_id
    
    # Training arguments
    training_args = TrainingArguments(
        output_dir=f"{OUTPUT_DIR}/{model_name}_{dataset_name}",
        num_train_epochs=NUM_EPOCHS,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        learning_rate=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        greater_is_better=True,
        logging_dir=f"{OUTPUT_DIR}/logs/{model_name}_{dataset_name}",
        logging_steps=100,
        report_to="none",  # Disable wandb
        fp16=use_fp16,
        bf16=use_bf16,
        gradient_accumulation_steps=2 if use_quantization else 1,
        warmup_ratio=0.1
    )
    
    # Data collator
    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
    
    # Initialize trainer
    trainer = WeightedTrainer(
        class_weights=class_weights,
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        tokenizer=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics
    )
    
    # Train
    print("Training...")
    trainer.train()
    
    # Evaluate on test set
    print("Evaluating on test set...")
    predictions = trainer.predict(test_dataset)
    preds = np.argmax(predictions.predictions, axis=1)
    labels = predictions.label_ids
    
    # Calculate metrics
    accuracy = accuracy_score(labels, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='weighted')
    
    metrics = {
        'accuracy': float(accuracy),
        'precision': float(precision),
        'recall': float(recall),
        'f1': float(f1)
    }
    
    # Print results
    print(f"\n{'-'*40}")
    print(f"Results for {model_name} on {dataset_name}:")
    print(f"  Accuracy:  {accuracy:.4f}")
    print(f"  Precision: {precision:.4f}")
    print(f"  Recall:    {recall:.4f}")
    print(f"  F1 Score:  {f1:.4f}")
    print(f"{'-'*40}")
    
    # Classification report
    report = classification_report(labels, preds, output_dict=True)
    print("\nClassification Report:")
    print(classification_report(labels, preds))
    
    # Confusion matrix
    cm_path = f"{OUTPUT_DIR}/{model_name}_{dataset_name}_confusion_matrix.png"
    plot_confusion_matrix(
        labels, preds,
        labels=list(range(num_labels)),
        title=f"Confusion Matrix: {model_name} on {dataset_name}",
        save_path=cm_path
    )
    
    # Save results
    results = save_results(model_name, dataset_name, metrics, report, OUTPUT_DIR)
    
    # Clean up
    del model, trainer
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    
    return results

---
# Train Individual Models

Run each cell below to train and evaluate a specific model. You can run them one by one to manage GPU memory.

---

## 1. DistilBERT

In [ ]:
distilbert_results = train_and_evaluate_model(
    model_name='distilbert',
    model_path=MODELS['distilbert'],
    train_df=train_df,
    val_df=val_df,
    test_df=test_df,
    num_labels=num_labels,
    class_weights=class_weights,
    dataset_name=DATASET_NAME,
    use_quantization=False
)

## 2. DeBERTa

In [9]:
deberta_results = train_and_evaluate_model(
    model_name='deberta',
    model_path=MODELS['deberta'],
    train_df=train_df,
    val_df=val_df,
    test_df=test_df,
    num_labels=num_labels,
    class_weights=class_weights,
    dataset_name=DATASET_NAME,
    use_quantization=False
)


Training: DEBERTA on DAILY
Tokenizing data...


Map: 100%|██████████| 1000/1000 [00:00<00:00, 9178.81 examples/s]


Loading model...


Some weights of DebertaForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-base and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Precision mode: fp32


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 2, 'bos_token_id': 1}.


Training...


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

## 3. RoBERTa

In [29]:
roberta_results = train_and_evaluate_model(
    model_name='roberta',
    model_path=MODELS['roberta'],
    train_df=train_df,
    val_df=val_df,
    test_df=test_df,
    num_labels=num_labels,
    class_weights=class_weights,
    dataset_name=DATASET_NAME,
    use_quantization=False
)


Training: ROBERTA on EMPATHETIC
Tokenizing data...


Map: 100%|██████████| 6460/6460 [00:00<00:00, 22588.45 examples/s]


Loading model...


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Precision mode: bf16 (auto)
Training...


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

## 4. BanglaBERT
> **Note:** BanglaBERT is pre-trained on Bengali. Performance on English datasets may be limited.

In [ ]:
banglabert_results = train_and_evaluate_model(
    model_name='banglabert',
    model_path=MODELS['banglabert'],
    train_df=train_df,
    val_df=val_df,
    test_df=test_df,
    num_labels=num_labels,
    class_weights=class_weights,
    dataset_name=DATASET_NAME,
    use_quantization=False
)

## 5. BanglishBERT
> **Note:** BanglishBERT is pre-trained on Bangla-English code-mixed text. Performance on pure English may vary.

In [ ]:
banglishbert_results = train_and_evaluate_model(
    model_name='banglishbert',
    model_path=MODELS['banglishbert'],
    train_df=train_df,
    val_df=val_df,
    test_df=test_df,
    num_labels=num_labels,
    class_weights=class_weights,
    dataset_name=DATASET_NAME,
    use_quantization=False
)

## 6. XLNet

In [28]:
xlnet_results = train_and_evaluate_model(
    model_name='xlnet',
    model_path=MODELS['xlnet'],
    train_df=train_df,
    val_df=val_df,
    test_df=test_df,
    num_labels=num_labels,
    class_weights=class_weights,
    dataset_name=DATASET_NAME,
    use_quantization=False
)


Training: XLNET on EMPATHETIC
Tokenizing data...


Map: 100%|██████████| 6460/6460 [00:00<00:00, 17496.42 examples/s]


Loading model...


Some weights of XLNetForSequenceClassification were not initialized from the model checkpoint at xlnet-base-cased and are newly initialized: ['logits_proj.bias', 'logits_proj.weight', 'sequence_summary.summary.bias', 'sequence_summary.summary.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Precision mode: bf16 (auto)
Training...


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

## 7. Falcon-7B (Quantized with QLoRA)
> **Note:** This model requires significant GPU memory. Uses 4-bit quantization and LoRA for efficient training.

In [ ]:
falcon_results = train_and_evaluate_model(
    model_name='falcon',
    model_path=MODELS['falcon'],
    train_df=train_df,
    val_df=val_df,
    test_df=test_df,
    num_labels=num_labels,
    class_weights=class_weights,
    dataset_name=DATASET_NAME,
    use_quantization=True  # Enable 4-bit quantization
)

---
# Summary

After running all models, check the `results/` folder for:
- JSON files with detailed metrics
- Confusion matrix images

Use the `compare_results.ipynb` notebook to generate comparison charts.

---